In [ ]:
!pip install -q torch transformers matplotlib seaborn koreanize-matplotlib openai

In [ ]:
# ============================================================
# 실습 2-1. Decoder-only Transformer 구현 및 Temperature 실습
# 목표: Causal Mask, Decoder 구조, Temperature 효과를 체험한다
# 사전 설치: pip install torch transformers openai matplotlib seaborn koreanize-matplotlib
# ============================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os

plt.rcParams['axes.unicode_minus'] = False

# ============================================================
# [실습 변수] 여기만 수정하여 다양한 실험 가능
# ============================================================

# Causal Mask 시각화용 토큰 목록
# Self-Attention 실습과 같은 문장을 사용하면 비교 효과 높음
CAUSAL_TOKENS = ["연구", "데이터는", "체계적으로", "관리", "되어야", "한다"]

# Decoder-only 모델 구성 (소형 시연용)
VOCAB_SIZE = 1000   # 어휘 크기
MAX_LEN    = 64     # 최대 시퀀스 길이
D_MODEL    = 64     # 임베딩 차원 (실습 1과 동일하게 맞추는 것을 권장)
N_HEADS    = 4      # Multi-Head Attention 헤드 수 (D_MODEL의 약수여야 함)
FFN_DIM    = 128    # Feed-Forward Network 중간 차원 (보통 D_MODEL * 4)
N_LAYERS   = 2      # Decoder 레이어 반복 횟수

# Temperature 실험용 프롬프트
PROMPT = "연구 데이터는 반드시"

# 비교할 Temperature 값 목록
# 0에 가까울수록 결정론적, 높을수록 창의적/불규칙
TEMPERATURES = [0.1, 0.7, 1.5]

# 각 Temperature에서 생성할 샘플 수
N_SAMPLES = 3

# 생성할 최대 토큰 수
MAX_NEW_TOKENS = 80

# OpenAI 모델명
OPENAI_MODEL = "gpt-4o-mini"

# ============================================================
# 이하 코드는 수정 불필요
# ============================================================

# ── API 키 설정 ──────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab Secrets에서 API 키 로드 완료")
except Exception:
    pass
# os.environ["OPENAI_API_KEY"] = "sk-..."
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "API 키가 설정되지 않았습니다.\n"
        "방법 A: 왼쪽 사이드바 열쇠 아이콘 -> 'OPENAI_API_KEY' 등록\n"
        "방법 B: os.environ['OPENAI_API_KEY'] = 'sk-...' 주석 해제"
    )
from openai import OpenAI
api_client = OpenAI()
print(f"API 키 확인: sk-...{os.environ['OPENAI_API_KEY'][-4:]}")


# ── 블록 1: Causal Mask 시각화 ───────────────────────────────
# Causal Mask: Decoder는 미래 토큰을 볼 수 없음
# 상삼각 행렬에서 True인 위치(대각선 위쪽)가 차단됨
# 이유: 학습 시 "미래 정답"을 보고 예측하면 의미 없음

def causal_mask(seq_len):
    # torch.triu: 상삼각 행렬 생성
    # diagonal=1: 대각선 바로 위부터 차단 (자기 자신은 참조 가능)
    return torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

seq_len = len(CAUSAL_TOKENS)
mask_np = causal_mask(seq_len).numpy().astype(float)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Encoder: 양방향 참조 가능 (BERT 계열)
sns.heatmap(
    np.zeros((seq_len, seq_len)),
    xticklabels=CAUSAL_TOKENS, yticklabels=CAUSAL_TOKENS,
    cmap="Blues", vmin=0, vmax=1,
    linewidths=0.5, ax=axes[0], cbar=False
)
axes[0].set_title("Encoder Self-Attention\n(양방향 - 모든 토큰 참조 가능)", fontsize=11)
axes[0].tick_params(axis='x', rotation=35)

# Decoder: 단방향 참조 (GPT 계열)
cmap_m = sns.color_palette("Blues", as_cmap=True)
cmap_m.set_bad(color='#F5C4B3')   # 차단된 위치: 연한 빨강
sns.heatmap(
    np.where(mask_np == 1, np.nan, 0.6),
    xticklabels=CAUSAL_TOKENS, yticklabels=CAUSAL_TOKENS,
    cmap=cmap_m, vmin=0, vmax=1,
    linewidths=0.5, ax=axes[1], cbar=False
)
axes[1].set_title("Decoder Causal Attention\n(단방향 - 빨강=차단)", fontsize=11)
axes[1].tick_params(axis='x', rotation=35)

plt.suptitle("Encoder vs Decoder Attention 마스크 비교", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("causal_mask.png", dpi=150, bbox_inches='tight')
plt.show()
print("Decoder는 이전 토큰만 참조 -> 다음 토큰 예측 (자기회귀 생성)")


# ── 블록 2: Decoder-only Transformer 직접 구현 ───────────────

class CausalSelfAttention(nn.Module):
    """
    Causal Self-Attention:
    - Multi-Head로 여러 관점에서 동시에 Attention 계산
    - Causal Mask로 미래 토큰 차단
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model은 n_heads의 배수여야 합니다"
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads  # 헤드당 차원
        self.d_model = d_model
        # 입력을 Q, K, V로 변환하는 가중치 행렬
        self.wq = nn.Linear(d_model, d_model, bias=False)
        self.wk = nn.Linear(d_model, d_model, bias=False)
        self.wv = nn.Linear(d_model, d_model, bias=False)
        self.wo = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, S, _ = x.shape
        # 입력을 헤드별로 분리: (B, S, d_model) -> (B, n_heads, S, d_head)
        q = self.wq(x).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        k = self.wk(x).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        v = self.wv(x).view(B, S, self.n_heads, self.d_head).transpose(1, 2)

        # Scaled Dot-Product Attention
        scores  = torch.matmul(q, k.transpose(-2, -1)) / (self.d_head ** 0.5)
        # Causal Mask 적용: 미래 위치를 -inf로 설정 -> softmax 후 0이 됨
        scores  = scores.masked_fill(causal_mask(S).to(x.device), float('-inf'))
        weights = F.softmax(scores, dim=-1)
        # 헤드별 출력을 합쳐서 원래 차원으로 복원
        ctx     = torch.matmul(weights, v).transpose(1, 2).contiguous()
        return self.wo(ctx.view(B, S, self.d_model)), weights


class DecoderBlock(nn.Module):
    """
    Decoder 블록 = CausalSelfAttention + FFN + 잔차 연결 + LayerNorm
    잔차 연결: x = x + sublayer(x) -> 학습 안정성 향상
    LayerNorm: 각 토큰의 벡터를 정규화 -> 학습 안정성 향상
    """
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attn = CausalSelfAttention(d_model, n_heads)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),           # 활성화 함수: ReLU의 부드러운 버전
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout), # 과적합 방지
        )
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, w = self.attn(x)
        # 잔차 연결: Attention 출력을 입력에 더함
        x = self.ln1(x + self.drop(attn_out))
        # 잔차 연결: FFN 출력을 입력에 더함
        x = self.ln2(x + self.ffn(x))
        return x, w


class DecoderOnlyTransformer(nn.Module):
    """
    Decoder-only Transformer (GPT 계열과 동일한 구조)
    입력: 토큰 ID 시퀀스
    출력: 각 위치에서 다음 토큰의 확률 분포 (logits)
    """
    def __init__(self, vocab_size, max_len, d_model, n_heads, ffn_dim, n_layers):
        super().__init__()
        # 토큰 임베딩: ID -> 벡터
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        # 위치 임베딩: 위치 ID -> 벡터 (Positional Encoding의 학습 버전)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.blocks    = nn.ModuleList(
            [DecoderBlock(d_model, n_heads, ffn_dim) for _ in range(n_layers)]
        )
        self.ln_final  = nn.LayerNorm(d_model)
        # LM Head: 벡터 -> 어휘 크기의 로짓 (다음 토큰 확률)
        self.lm_head   = nn.Linear(d_model, vocab_size, bias=False)
        self.scale     = d_model ** 0.5  # 임베딩 스케일링

    def forward(self, token_ids):
        B, S    = token_ids.shape
        pos_ids = torch.arange(S, device=token_ids.device).unsqueeze(0)
        # 토큰 의미 벡터 + 위치 벡터를 더해 최종 입력 생성
        x       = (self.tok_embed(token_ids) + self.pos_embed(pos_ids)) * self.scale
        all_w   = []
        for block in self.blocks:
            x, w = block(x)
            all_w.append(w)
        return self.lm_head(self.ln_final(x)), all_w

# 모델 생성 및 구조 확인
model_custom = DecoderOnlyTransformer(
    vocab_size=VOCAB_SIZE, max_len=MAX_LEN,
    d_model=D_MODEL, n_heads=N_HEADS,
    ffn_dim=FFN_DIM, n_layers=N_LAYERS
)
with torch.no_grad():
    dummy  = torch.ones(1, 10, dtype=torch.long)
    logits, attn_w = model_custom(dummy)

total_params = sum(p.numel() for p in model_custom.parameters())
print(f"\n출력 shape : logits={tuple(logits.shape)}")
print(f"             -> (batch=1, seq_len=10, vocab_size={VOCAB_SIZE})")
print(f"Attention  : {len(attn_w)} layers, {tuple(attn_w[0].shape)} each")
print(f"             -> (batch, n_heads={N_HEADS}, seq_len, seq_len)")
print(f"파라미터 수: {total_params:,}")
print(f"  참고: GPT-2는 약 1억 2천만, GPT-3는 약 1750억 파라미터")


# ── 블록 3: Temperature 실험 (OpenAI API) ────────────────────
def generate_with_temperature(prompt, temperature, max_new=MAX_NEW_TOKENS,
                               n_samples=N_SAMPLES):
    """
    OpenAI API로 temperature 실험
    temperature 원리 (실습 1-3 PE와 연결):
      - 모델의 logit을 temperature로 나눔
      - temperature < 1: logit 차이 확대 -> 높은 확률 토큰에 더 집중
      - temperature > 1: logit 차이 축소 -> 확률 분포 평탄화
    """
    results = []
    for _ in range(n_samples):
        resp = api_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=float(temperature),
            max_tokens=max_new,
        )
        results.append(resp.choices[0].message.content.strip())
    return results

# 온도별 생성 결과 비교
print("\n" + "=" * 60)
print(f"프롬프트: '{PROMPT}'")
print("=" * 60)

temp_results = {}
for temp in TEMPERATURES:
    samples = generate_with_temperature(PROMPT, temperature=temp)
    temp_results[temp] = samples
    label = {
        0.1: "낮은 온도 (결정론적 - 항상 비슷한 답변)",
        0.7: "적정 온도 (자연스러운 다양성)",
        1.5: "높은 온도 (창의적이지만 불규칙)"
    }.get(temp, "")
    print(f"\n[Temperature = {temp}]  {label}")
    for i, s in enumerate(samples, 1):
        print(f"  샘플 {i}: {s[:120]}...")

# Temperature 원리 시각화
def temperature_softmax(logits_raw, temperature):
    """temperature를 나눠 softmax를 적용하는 함수"""
    scaled = np.array(logits_raw) / max(temperature, 1e-5)
    exp    = np.exp(scaled - np.max(scaled))
    return exp / exp.sum()

vocab_sample = ["관리되어야", "공개되어야", "보존되어야", "분석되어야"]
logits_raw   = np.array([3.0, 2.5, 1.0, 0.5])  # 가상의 logit 값
temps_plot   = [0.1, 0.5, 1.0, 1.5, 2.0]
colors       = ['#E24B4A', '#EF9F27', '#1D9E75', '#378ADD', '#7F77DD']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x_pos = np.arange(len(vocab_sample))
w     = 0.15
for i, (t, c) in enumerate(zip(temps_plot, colors)):
    probs = temperature_softmax(logits_raw, t)
    axes[0].bar(x_pos + i * w, probs, width=w, label=f"T={t}", color=c, alpha=0.85)
axes[0].set_xticks(x_pos + w * 2)
axes[0].set_xticklabels(vocab_sample)
axes[0].set_ylabel("확률")
axes[0].set_title("Temperature별 다음 토큰 확률 분포\n(T 낮을수록 첫 토큰에 집중)", fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1.05)

# 엔트로피 곡선: Temperature가 높을수록 불확실성(다양성) 증가
entropies = []
for t in np.linspace(0.1, 2.5, 50):
    p = temperature_softmax(logits_raw, t)
    entropies.append(-np.sum(p * np.log(p + 1e-9)))
axes[1].plot(np.linspace(0.1, 2.5, 50), entropies, color='#378ADD', linewidth=2)
axes[1].axvline(1.0, color='#1D9E75', linestyle='--', label='T=1 (기본값)')
axes[1].set_xlabel("Temperature")
axes[1].set_ylabel("엔트로피 (다양성)")
axes[1].set_title("Temperature 높을수록 다양성 증가\n(엔트로피 상승)", fontsize=11)
axes[1].legend(fontsize=9)

plt.suptitle("Temperature: 생성 다양성을 조절하는 파라미터", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("temperature_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nTemperature 정리:")
print("  낮은 T (0.1~0.3) -> 요약, 사실 추출  (일관성 중요)")
print("  적정 T (0.5~0.9) -> 일반 대화, 설명 생성")
print("  높은 T (1.0~)    -> 가설 생성, 아이디어 브레인스토밍")


# ── 블록 4: GPT-2 내부 Attention 시각화 ─────────────────────
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2Config

print("\nAttention 시각화용 GPT-2 로드 중...")
# output_attentions=True: 모든 레이어의 Attention 가중치를 반환
config_vis = GPT2Config.from_pretrained("gpt2", output_attentions=True)
tok_vis    = AutoTokenizer.from_pretrained("gpt2")
model_vis  = GPT2LMHeadModel.from_pretrained("gpt2", config=config_vis)
model_vis.eval()
print("로드 완료")

vis_prompt = "The research data should be"
enc_vis    = tok_vis(vis_prompt, return_tensors="pt", return_token_type_ids=False)
enc_vis["attention_mask"] = torch.ones_like(enc_vis["input_ids"])

with torch.no_grad():
    out_vis = model_vis(
        input_ids=enc_vis["input_ids"],
        attention_mask=enc_vis["attention_mask"],
    )

attn_layers = [a for a in out_vis.attentions if a is not None]
last_attn   = attn_layers[-1].detach().numpy()  # (1, n_heads, seq, seq)
head_0      = last_attn[0, 0]                   # 첫 번째 헤드

tok_labels  = [tok_vis.decode([tid]).strip()
               for tid in enc_vis["input_ids"][0].tolist()]

print(f"Attention shape: {last_attn.shape}  ->  (batch, n_heads, seq, seq)")
print(f"토큰: {tok_labels}")

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    head_0,
    xticklabels=tok_labels, yticklabels=tok_labels,
    cmap="YlOrRd", annot=True, fmt=".2f",
    linewidths=0.5, ax=ax
)
ax.set_title(
    f"GPT-2 마지막 레이어 Head-0 Attention\n프롬프트: '{vis_prompt}'",
    fontsize=11
)
ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.savefig("gpt2_attention.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n오전 직접 구현 Attention vs GPT-2 실제 Attention - 같은 구조")
print("내일 오전 OpenAI API는 이 GPT 계열의 서비스 버전")